# Universal Grammar Generative Model

In [1]:
INSTALL = false

if INSTALL
    using Pkg
    Pkg.add("StatsPlots"); using StatsPlots
    Pkg.activate("psyc261")
    Pkg.add(["JLD2", "Distributions", "ProgressMeter", "Parameters", 
            "Random", "Gen", "Plots", "PyCall", "Conda", "JSON3"])
end

In [2]:
using Pkg
Pkg.activate("psyc261")
using JLD2
using Distributions
using ProgressMeter
using Parameters
using Random
using StatsPlots
using Gen, Plots
using PyCall
np = pyimport("numpy")

include("../src/ug_model.jl")
using .UGmodel

  Activating project at `~/Algorithms-of-the-Mind/aom/notebooks/psyc261`


In [3]:
# Lexicon constants
const NOUNS = UGmodel.LEXICON.nouns
const VERBS_TRANS = UGmodel.LEXICON.verbs_trans
const VERBS_INTR = UGmodel.LEXICON.verbs_intr
const ADJECTIVES = UGmodel.LEXICON.adjectives
const WH_WORD = UGmodel.LEXICON.wh_word
const COMPL = UGmodel.LEXICON.complementizer

# Vocab construction
const VOCAB = unique(vcat(["PAD", WH_WORD, COMPL], NOUNS, VERBS_TRANS, VERBS_INTR, ADJECTIVES))
const VOCAB_SIZE = length(VOCAB)
const WORD_TO_IDX = Dict(w => i for (i,w) in enumerate(VOCAB))
const MAX_SENT_LEN = 12

# IDs for vocab
const NOUN_IDS = [WORD_TO_IDX[w] for w in NOUNS]
const VERB_TRANS_IDS = [WORD_TO_IDX[w] for w in VERBS_TRANS]
const VERB_INTR_IDS = [WORD_TO_IDX[w] for w in VERBS_INTR]
const ADJ_IDS = [WORD_TO_IDX[w] for w in ADJECTIVES]
const WH_ID = WORD_TO_IDX[WH_WORD]
const COMPL_ID = WORD_TO_IDX[COMPL]
const PAD_ID = WORD_TO_IDX["PAD"]

# Helper to define all langs
function all_language_types()
    languages = NamedTuple[]
    for hd in [false, true]
        for sd in [false, true]
            for ao in [false, true]
                for wh in [false, true]
                    push!(languages, (hd=hd, sd=sd, ao=ao, wh=wh))
                end
            end
        end
    end
    return languages
end

const ALL_LANGUAGES = all_language_types()

# Math helpers
σ(x) = tanh.(x)
sigmoid(x) = 1.0 ./ (1.0 .+ exp.(-x))

# Other helpers
# Maps vectors of IDs back to vectors of Strings
function decode_sentences(sentences_ids)
    decoded = Vector{Vector{String}}()
    for sent_ids in sentences_ids
        push!(decoded, [VOCAB[idx] for idx in sent_ids])
    end
    return decoded
end

decode_sentences (generic function with 1 method)

# Generative Model

To model the Principles & Parameters prior, we are using the following UG parameters:

- Head directionality: in this case, we're using it to control whether the verb or the object will come first
- Subject drop: whether the subject of a sentence has to be kept explicit or if it can be dropped and understood from the context
- Adjective order: whether the noun or the adjective comes first
- Wh-fronting: in a Wh- question (what, who, which etc.), does the wh- word stay in its place (in-situ) or does it go to the front of the sentence

We then make the distinction for three different types of utterances: 
1. Transitive declarative: a phrase containing a subject, transitive verb, and its object in a declarative manner (no wh- word)
2. Intransitive: a phrase containing a subject and an intransitive verb
3. Transitive question: a phrase containing a subject, transitive verb, its object and a wh- word that can be fronted or not

We establish Bernoulli priors over each one of the parameters, and also a small chance that they might "slip" and occasionally be flipped in a given sentence. Although this is not the case for human languages, it suffices as a means to test our hypothesis.

The final sentence is then produced taking into account the noisy parameters and emission probability.

To construct the sentence, the model follows a hierarchical process:

1.  Recursive Noun Phrases (NP): It builds NPs using the `make_np` helper. This allows for recursion via relative clauses (e.g., "The dog [that bit the cat]..."), which is a fundamental property of human language.
2.  Verb Phrases (VP): It combines the verb and the object, respecting the Head Direction parameter (VO vs. OV).
3.  Clause Construction: Finally, it assembles the Subject and VP, applying Subject Drop and handling Wh-movement (checking if the wh- word moves to the front or stays in-situ).

In [4]:
const NOISE_PROB = 0.1

function emission_probs(target_idx::Int; V::Int = VOCAB_SIZE, epsilon::Float64 = 0.05)
    @assert 0 < epsilon < 1
    probs = fill(epsilon / (V-1), V)
    probs[target_idx] = 1.0 -epsilon
    return probs
end

const EMISSION_CACHE = [emission_probs(i) for i in 1:VOCAB_SIZE]

13-element Vector{Vector{Float64}}:
 [0.95, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667]
 [0.004166666666666667, 0.95, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667]
 [0.004166666666666667, 0.004166666666666667, 0.95, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667]
 [0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.95, 0.004166666666666667, 0.004166666666666667, 0.004166666666666667, 0.004166

In [5]:
@gen function sentence_kernel(hd::Bool, sd::Bool, ao::Bool, wh::Bool)
    # Decide if the sentence is going to have an embedded clause or not
    is_embedded = @trace(bernoulli(0.3), :is_embedded)
    
    # Utterance type: 1=trans declarative, 2=intransitive, 3=trans question
    utt_type = @trace(categorical([0.4, 0.3, 0.3]), :utt_type)

    # Sample lexical items
    subj_id = NOUN_IDS[@trace(categorical(fill(1/length(NOUNS), length(NOUNS))), :subj)]
    obj_id = NOUN_IDS[@trace(categorical(fill(1/length(NOUNS), length(NOUNS))), :obj)]
    adj_id = ADJ_IDS[@trace(categorical(fill(1/length(ADJECTIVES), length(ADJECTIVES))), :adj)]

    # Verb selection
    if utt_type == 1 || utt_type == 3
        v_idx = @trace(categorical(fill(1/length(VERB_TRANS_IDS), length(VERB_TRANS_IDS))), :v_trans)
        verb_id = VERB_TRANS_IDS[v_idx]
    else
        v_idx = @trace(categorical(fill(1/length(VERB_INTR_IDS), length(VERB_INTR_IDS))), :v_intr)
        verb_id = VERB_INTR_IDS[v_idx]
    end

    function make_np(noun, adj, adj_order_first, head_direction_first, allow_rel_clause, addr_key)
        base_np = adj_order_first ? [adj, noun] : [noun, adj]
        
        # Recursive relative clause logic
        if allow_rel_clause && is_embedded
            # Sample a verb for the embedded phrase ("The dog that ate")
            rel_v_idx = @trace(categorical(fill(1/length(VERB_INTR_IDS), length(VERB_INTR_IDS))), addr_key)
            rel_verb = VERB_INTR_IDS[rel_v_idx]
            
            # Construct clause "that" + verb (simplified relative clause)
            rel_phrase = head_direction_first ? [COMPL_ID, rel_verb] : [rel_verb, COMPL_ID]
            
            # Combine base NP with relative clause
            return head_direction_first ? vcat(base_np, rel_phrase) : vcat(rel_phrase, base_np)
        else
            return base_np
        end
    end

    # Adjective order error
    ao_slip = @trace(bernoulli(NOISE_PROB), :ao_slip)
    effective_ao = ao_slip ? !ao : ao

    # Head direction error
    hd_slip = @trace(bernoulli(NOISE_PROB), :hd_slip)
    effective_hd = hd_slip ? !hd : hd

    subj_np = make_np(subj_id, adj_id, effective_ao, effective_hd, true, :rel_verb_subj)
    obj_np = make_np(obj_id, adj_id, effective_ao, effective_hd, true, :rel_verb_obj)

    # VP Construction
    vp = Vector{Int}()
    if utt_type == 1 || utt_type == 3
        vp = (effective_hd) ? vcat([verb_id], obj_np) : vcat(obj_np, [verb_id])
    else
        vp = [verb_id]
    end

    # Subject drop error
    sd_slip = @trace(bernoulli(NOISE_PROB), :sd_slip)
    effective_sd = sd_slip ? !sd : sd
    
    # Drop logic
    drop_subj = (effective_sd) ? @trace(bernoulli(0.5), :drop_subj) : false

    clause = Vector{Int}()
    
    if !drop_subj
        append!(clause, subj_np)
    end
    append!(clause, vp)

    # Wh-fronting logic
    if utt_type == 3
        wh_slip = @trace(bernoulli(NOISE_PROB), :wh_slip)
        effective_wh = wh_slip ? !wh : wh
        
        if effective_wh
            pushfirst!(clause, WH_ID)
        else
            push!(clause, WH_ID)
        end
    end

    final_sent = zeros(Int, MAX_SENT_LEN)
    for pos in 1:MAX_SENT_LEN
        token_id = pos <= length(clause) ? clause[pos] : PAD_ID
        final_sent[pos] = token_id
        probs = EMISSION_CACHE[token_id]
        
        @trace(categorical(probs), (:token, pos))
    end

    return clause
end

DynamicDSLFunction{Any}(Dict{Symbol, Any}(), Dict{Symbol, Any}(), Type[Bool, Bool, Bool, Bool], false, Union{Nothing, Some{Any}}[nothing, nothing, nothing, nothing], var"##sentence_kernel#292", Bool[0, 0, 0, 0], false)

### The Corpus Prior

While sentence_kernel generates a single noisy utterance, this function defines the generative model for the entire dataset (the "Language"). 

It samples the True Grammar $\theta$ (the four binary parameters) once, and then generates a batch of $N$ sentences based on that specific grammar. This models the linguistic environment the learner observes.

In [6]:
const sentences_map = Gen.Map(sentence_kernel)

@gen function corpus_model(num_sentences::Int)
    # Universal Grammar paramaters (theta)
    hd = @trace(bernoulli(0.5), :head_direction)
    sd = @trace(bernoulli(0.5), :subject_drop)
    ao = @trace(bernoulli(0.5), :adj_order)
    wh = @trace(bernoulli(0.5), :wh_fronting)

    hds = fill(hd, num_sentences)
    sds = fill(sd, num_sentences)
    aos = fill(ao, num_sentences)
    whs = fill(wh, num_sentences)

    sentences = @trace(sentences_map(hds, sds, aos, whs), :corpus)

    return sentences
end

DynamicDSLFunction{Any}(Dict{Symbol, Any}(), Dict{Symbol, Any}(), Type[Int64], false, Union{Nothing, Some{Any}}[nothing], var"##corpus_model#293", Bool[0], false)

We can generate a sample from our generative model. The trace contains the parameters used, and then the generated sentences.

In [7]:
trace = simulate(corpus_model, (10,))
sentences_ids = get_retval(trace)

theta = (
    hd = trace[:head_direction],
    sd = trace[:subject_drop],
    ao = trace[:adj_order],
    wh = trace[:wh_fronting]
)
println("Theta: ", theta)

sentences_words = decode_sentences(sentences_ids)

for (i, sent) in enumerate(sentences_words)
    println("Sentence $i: ", sent)
end

Theta: (hd = false, sd = false, ao = true, wh = true)
Sentence 1: ["dog", "big", "dog", "big", "see"]
Sentence 2: ["run", "that", "big", "fox", "sleep"]
Sentence 3: ["sleep", "that", "small", "fox", "run"]
Sentence 4: ["what", "sleep", "that", "big", "fox", "run", "that", "big", "dog", "see"]
Sentence 5: ["big", "fox", "big", "cat", "see"]
Sentence 6: ["what", "small", "dog", "small", "fox", "chase"]
Sentence 7: ["big", "fox", "that", "run", "see", "big", "dog", "that", "run"]
Sentence 8: ["what", "big", "fox", "big", "dog", "attack"]
Sentence 9: ["small", "fox", "small", "dog", "attack", "what"]
Sentence 10: ["small", "fox", "small", "cat", "attack"]


# Neural Network (Amortized Inference)

Here we define the "Innate Solver," a neural network designed to act as a fast heuristic guide for the probabilistic model.

* Architecture: A simple feed-forward network with one hidden layer ($H=128$).
* Goal: It takes a sequence of sentences (one-hot encoded) and predicts the probability of each grammar parameter (Head Direction, Subject Drop, etc.).
* Amortization: Instead of performing expensive Bayesian computation from scratch for every new language, we train this network to instantly "guess" the parameters, which we will later use to initialize our particle filter.

In [8]:
const NUM_SENTS_TRAIN = 10
const INPUT_DIM = VOCAB_SIZE * MAX_SENT_LEN * NUM_SENTS_TRAIN
const H = 128

@gen function innate_solver(input_vector::AbstractVector{<:Real})
    @param W1::Matrix{Float64}
    @param b1::Vector{Float64}
    @param W2::Matrix{Float64}
    @param b2::Vector{Float64}

    hidden_layer = σ(W1 * input_vector + b1)
    output = W2 * hidden_layer + b2

    probs = sigmoid(output)

    for i in 1:4
        @trace(Gen.bernoulli(probs[i]), (:param, i))
    end
    
    return probs
end

init_weight(out, inc) = randn(out, inc) * 0.01
init_param!(innate_solver, :W1, init_weight(H, INPUT_DIM))
init_param!(innate_solver, :b1, zeros(H))
init_param!(innate_solver, :W2, init_weight(4, H))
init_param!(innate_solver, :b2, zeros(4))

4-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0

In [9]:
function predict_nn(input_vec::Vector{Float64}, params)
    hidden = σ(params.W1 * input_vec + params.b1)
    output = params.W2 * hidden + params.b2
    probs = sigmoid(output)

    return probs
end

predict_nn (generic function with 1 method)

In [10]:
const TRAIN_MODEL = false
const PARAMS_FILE = "innate_solver_params.jld2"
const NUM_EPOCHS = 500
const EPOCH_SIZE = 50
const LEARNING_RATE = 0.001

function encode_corpus(sentences, num_sents)
    encoded = zeros(Float64, INPUT_DIM)

    cursor = 0
    for i in 1:num_sents
        if i > length(sentences) break end

        sent = sentences[i]
        for j in 1:MAX_SENT_LEN
            token_id = 0
            
            token_id = j <= length(sent) ? sent[j] : PAD_ID

            encoded[cursor * VOCAB_SIZE + token_id] = 1.0
            cursor += 1
        end
    end

    return encoded
end

function save_nn_params(losses, filename=PARAMS_FILE)
    W1 = get_param(innate_solver, :W1)
    b1 = get_param(innate_solver, :b1)
    W2 = get_param(innate_solver, :W2)
    b2 = get_param(innate_solver, :b2)
    
    @save filename W1 b1 W2 b2 losses
    println("Parameters and training history saved to $filename")
end

function load_nn_params(filename=PARAMS_FILE)
    @load filename W1 b1 W2 b2 losses
    
    init_param!(innate_solver, :W1, W1)
    init_param!(innate_solver, :b1, b1)
    init_param!(innate_solver, :W2, W2)
    init_param!(innate_solver, :b2, b2)
    
    println("Parameters loaded from $filename")
    return losses
end

load_nn_params (generic function with 2 methods)

### Training Loop

We use **Supervised Learning** to train the solver. The training loop generates synthetic languages (drawing from the Generative Model defined above) to create infinite training data. The network minimizes the difference between its predictions and the "true" parameters of the synthetic language.

In [11]:
function train_innate_solver(; epochs=NUM_EPOCHS, epoch_size=EPOCH_SIZE, lr=LEARNING_RATE)
    param_update = Gen.ParamUpdate(Gen.FixedStepGradientDescent(lr), innate_solver)
    losses = Float64[]

    println("Training innate solver...")
    flush(stdout)

    t_start = time()

    function supervised_training_batch_generator()
        tr = simulate(corpus_model, (NUM_SENTS_TRAIN,))
        sentences = get_retval(tr)
    
        input_vec = encode_corpus(sentences)
    
        constraints = Gen.choicemap()
        constraints[(:param, 1)] = tr[:head_direction]
        constraints[(:param, 2)] = tr[:subject_drop]
        constraints[(:param, 3)] = tr[:adj_order]
        constraints[(:param, 4)] = tr[:wh_fronting]
    
        return ((input_vec,), constraints)
    end

    for epoch in 1:epochs
        epoch_start = time()
        epoch_loss = 0.0
        
        for i in 1:epoch_size
            (args, constraints) = supervised_training_batch_generator()
            (trace, weight) = generate(innate_solver, args, constraints)
            
            Gen.accumulate_param_gradients!(trace)
            epoch_loss += -get_score(trace)
        end
        
        Gen.apply!(param_update)
        
        avg_loss = epoch_loss / epoch_size
        push!(losses, avg_loss)
        
        if epoch == 1 || epoch % 10 == 0
            epoch_time = round(time() - epoch_start, digits=2)
            println("Epoch $epoch/$epochs | Loss: $(round(avg_loss, digits=4)) | Time: $(epoch_time)s")
            flush(stdout)
        end
    end

    elapsed = time() - t_start
    println("Training complete in $(round(elapsed / 60, digits=1)) minutes.")

    save_nn_params(losses)
    
    return losses
end

train_innate_solver (generic function with 1 method)

In [ ]:
if TRAIN_MODEL
    losses = train_innate_solver()
    display(plot(losses, 
        label="Loss", 
        title="Training Innate Solver", 
        xlabel="Epochs", 
        ylabel="Negative Log Likelihood",
        linewidth=2, color=:purple
    ))
elseif isfile(PARAMS_FILE)
    losses = load_nn_params()
    println("Skipped training, loaded saved parameters.")
    display(plot(losses, 
        label="Loss", 
        title="Training Innate Solver", 
        xlabel="Epochs", 
        ylabel="Negative Log Likelihood",
        linewidth=2, color=:purple
    ))
else
    error("No saved model found. Set TRAIN=true to train the model first.")
end

# Inference Step: Neuro-Symbolic Integration

We now combine the Generative Model and the Neural Network using **Sequential Monte Carlo (SMC)**.

1.  **Helpers**: Functions to integrate raw integer IDs and Gen's `choicemap` constraints.
2.  **`nn_proposal`**: This is the key point. Instead of proposing grammar hypotheses blindly from the prior, we use the trained Neural Network to propose likely parameters given the observed sentences. This "guides" the particle filter toward high-probability regions much faster than random sampling.

In [ ]:
function make_constraints(theta::NamedTuple)
    constraints = Gen.choicemap()
    constraints[:head_direction] = theta.hd
    constraints[:subject_drop] = theta.sd
    constraints[:adj_order] = theta.ao
    constraints[:wh_fronting] = theta.wh
    return constraints
end

function make_token_obs_upto(trace, up_to_sentence::Int)
    obs = Gen.choicemap()
    for i in 1:up_to_sentence
        for j in 1:MAX_SENT_LEN
            val = trace[:corpus => i => (:token, j)]
            obs[:corpus => i => (:token, j)] = val
        end
    end
    return obs
end

function make_token_obs_for_sentence(trace, sentence_idx::Int)
    obs = Gen.choicemap()
    for j in 1:MAX_SENT_LEN
        val = trace[:corpus => sentence_idx => (:token, j)]
        obs[:corpus => sentence_idx => (:token, j)] = val
    end
    return obs
end

In [ ]:
@gen function nn_proposal(input_vec::Vector{Float64}, nn_params::NamedTuple)
    nn_probs = predict_nn(input_vec, nn_params)
    
    hd = @trace(bernoulli(nn_probs[1]), :head_direction)
    sd = @trace(bernoulli(nn_probs[2]), :subject_drop)
    ao = @trace(bernoulli(nn_probs[3]), :adj_order)
    wh = @trace(bernoulli(nn_probs[4]), :wh_fronting)
    
    return (hd=hd, sd=sd, ao=ao, wh=wh)
end

In [ ]:
function record_state!(history, state::Gen.ParticleFilterState)
    log_weights = Gen.get_log_weights(state)
    log_Z = Gen.logsumexp(log_weights)
    weights = exp.(log_weights .- log_Z)
    
    push!(history[:log_ml], state.log_ml_est)
    
    ess = Gen.effective_sample_size(log_weights)
    push!(history[:ess], ess)
    
    traces = Gen.get_traces(state)
    
    theta_probs = zeros(16)
    
    for (tr, w) in zip(traces, weights)
        theta = (
            hd = tr[:head_direction],
            sd = tr[:subject_drop],
            ao = tr[:adj_order],
            wh = tr[:wh_fronting]
        )
        idx = findfirst(t -> t == theta, ALL_LANGUAGES)
        theta_probs[idx] += w
    end
    
    push!(history[:posterior_probs], theta_probs)
    
    # MAP estimate
    map_idx = argmax(theta_probs)
    push!(history[:map_estimate], ALL_LANGUAGES[map_idx])
end

In [ ]:
function run_smc(
    trace,
    n_sentences::Int;
    n_particles::Int = 100,
    ess_threshold::Float64 = 0.5,
    use_nn_init::Bool = false,
    input_vec::Union{Vector{Float64}, Nothing} = nothing
)
    # Get observations for first sentence
    init_obs = make_token_obs_upto(trace, 1)
    nn_params = (
        W1 = get_param(innate_solver, :W1),
        b1 = get_param(innate_solver, :b1),
        W2 = get_param(innate_solver, :W2),
        b2 = get_param(innate_solver, :b2)
    )
    
    # Initialize particle filter
    if use_nn_init && input_vec !== nothing
        state = Gen.initialize_particle_filter(
            corpus_model, (1,), init_obs, nn_proposal, (input_vec, nn_params), n_particles
        )
    else
        state = Gen.initialize_particle_filter(
            corpus_model, (1,), init_obs, n_particles
        )
    end
    
    history = Dict(
        :map_estimate => NamedTuple[],
        :posterior_probs => Vector{Float64}[], 
        :ess => Float64[],
        :resampled => Bool[],
        :log_ml => Float64[]
    )
    
    record_state!(history, state)
    
    for t in 2:n_sentences
        new_obs = make_token_obs_for_sentence(trace, t)
        
        Gen.particle_filter_step!(
            state, (t,), (UnknownChange(),), new_obs                     
        )
        
        did_resample = Gen.maybe_resample!(state, ess_threshold=ess_threshold * n_particles)
        push!(history[:resampled], did_resample)
        
        record_state!(history, state)
    end
    
    return history
end

## Experimental Simulation

We compare two learners on the same data to validate the "Innate Solver" hypothesis:
1.  **Naive SMC**: A learner that samples grammar hypotheses randomly (from the prior).
2.  **Guided SMC**: A learner that uses the Neural Network's output to propose hypotheses.

The `run_trial_smc` function simulates a specific "ground truth" language and tracks the posterior probability of the correct grammar as the learner hears more sentences.

In [ ]:
function run_trial_smc(
    theta::NamedTuple; 
    n_sentences::Int = NUM_SENTS_TRAIN,
    n_particles::Int = 100
)
    trace, _ = generate(corpus_model, (n_sentences,), make_constraints(theta))
    sentences = get_retval(trace)
    input_vec = encode_corpus(sentences)

    naive_history = run_smc(trace, n_sentences; n_particles=n_particles, use_nn_init=false)
    guided_history = run_smc(trace, n_sentences; n_particles=n_particles, use_nn_init=true, input_vec=input_vec)
    
    nn_params = (
        W1 = get_param(innate_solver, :W1),
        b1 = get_param(innate_solver, :b1),
        W2 = get_param(innate_solver, :W2),
        b2 = get_param(innate_solver, :b2)
    )
    nn_probs = predict_nn(input_vec, nn_params)
    nn_preds = (
        hd = nn_probs[1] > 0.5,
        sd = nn_probs[2] > 0.5,
        ao = nn_probs[3] > 0.5,
        wh = nn_probs[4] > 0.5,
    )
    
    nn_correct = (nn_preds.hd == theta.hd && nn_preds.sd == theta.sd &&
                  nn_preds.ao == theta.ao && nn_preds.wh == theta.wh)
    
    return (
        theta = theta,
        n_sentences = n_sentences,
        naive_history = (
            posterior_probs = naive_history[:posterior_probs],
            map_estimate = naive_history[:map_estimate],
            ess = naive_history[:ess],
            log_ml = haskey(naive_history, :log_ml) ? naive_history[:log_ml] : Float64[]
        ),
        guided_history = (
            posterior_probs = guided_history[:posterior_probs],
            map_estimate = guided_history[:map_estimate],
            ess = guided_history[:ess],
            log_ml = haskey(guided_history, :log_ml) ? guided_history[:log_ml] : Float64[]
        ),
        nn_correct = nn_correct,
        nn_probs = nn_probs,
        nn_preds = nn_preds,
        sentences_ids = sentences
    )
end

In [ ]:
function compute_smc_metrics(history, target::NamedTuple, n_sentences::Int)
    posterior_probs = history.posterior_probs
    map_estimates = history.map_estimate
    
    # Convergence metrics
    correct = [est == target for est in map_estimates]
    ever_converged = any(correct)
    first_correct = findfirst(correct)
    final_correct = correct[end]
    
    # Target probability over time
    all_thetas = all_language_types()
    target_idx = findfirst(t -> t == target, all_thetas)
    target_prob_history = [probs[target_idx] for probs in posterior_probs]
    
    # Area under curve
    auc = sum(target_prob_history) / n_sentences
    
    # Threshold crossings
    idx_50 = findfirst(p -> p >= 0.5, target_prob_history)
    sentences_to_50 = idx_50 === nothing ? Inf : idx_50
    idx_90 = findfirst(p -> p >= 0.9, target_prob_history)
    sentences_to_90 = idx_90 === nothing ? Inf : idx_90
    
    max_prob_history = [maximum(probs) for probs in posterior_probs]
    
    return (
        ever_converged = ever_converged,
        first_correct = first_correct,
        final_correct = final_correct,
        final_target_prob = target_prob_history[end],
        target_prob_history = target_prob_history,
        auc = auc,
        sentences_to_50 = sentences_to_50,
        sentences_to_90 = sentences_to_90,
        max_prob_history = max_prob_history
    )
end

In [ ]:
function process_raw_trials(raw_trials)
    processed = []
    for t in raw_trials
        naive_metrics = compute_smc_metrics(t.naive_history, t.theta, t.n_sentences)
        guided_metrics = compute_smc_metrics(t.guided_history, t.theta, t.n_sentences)
        
        push!(processed, (
            naive_metrics = naive_metrics,
            guided_metrics = guided_metrics,
            nn_correct = t.nn_correct,
            nn_probs = t.nn_probs,
            nn_preds = t.nn_preds,
            theta = t.theta,
            sentences_ids = t.sentences_ids
        ))
    end
    return processed
end

function save_raw_trials(filename::String, raw_trials)
    @save filename raw_trials
    println("Saved $(length(raw_trials)) trials to $filename")
end

function load_and_process_trials(filename::String)
    @load filename raw_trials
    println("Loaded $(length(raw_trials)) raw trials from $filename")
    
    processed = process_raw_trials(raw_trials)
    println("Processed into analysis-ready format")
    
    return processed
end

In [ ]:
RUN_SMC = false

SMC_FILE = "smc_results_backup.jld2"
U_LEARNING_CURVE = "u_learning_results.jld2"

if RUN_SMC
    raw_trials = []
    for theta in all_language_types()
        for i in 1:5
            println("Iteration $i for theta=$theta")
            push!(raw_trials, run_trial_smc(theta; n_sentences=30, n_particles=200))
        end
        GC.gc()
    end
    save_raw_trials(SMC_FILE, raw_trials)

    all_trials = process_raw_trials(raw_trials)
else
    # @load U_LEARNING_CURVE all_trials
    all_trials = load_and_process_trials(SMC_FILE)
end

# Visualization and Analysis

Finally, we visualize the learning trajectories. The dashboard below plots:

1.  **Global Performance**:
    * **Learning Curves**: Average probability of the correct grammar over time.
    * **Convergence Speed**: The fraction of trials that reached >50% confidence at each time step.
    * **Summary Statistics**: Key metrics like AUC (area under curve) improvement and final accuracy.
2.  **Neural Network Diagnostics**:
    * **Per-Parameter Accuracy**: How well the NN predicts specific parameters (e.g., Head Direction).
    * **Parameter Learnability**: Which parameters are inherently harder for the system to learn.
3.  **Linguistic Analysis**:
    * **Language Difficulty**: Performance breakdown by specific language types (e.g., "VO-Drop-AN-Front").
    * **Typology**: A comparison of learning trajectories for "Natural" (attested) vs. "Rare" (unattested) language patterns.

In [ ]:
function compute_nn_metrics(trials)
    hd_acc = mean([t.nn_preds.hd == t.theta.hd for t in trials])
    sd_acc = mean([t.nn_preds.sd == t.theta.sd for t in trials])
    ao_acc = mean([t.nn_preds.ao == t.theta.ao for t in trials])
    wh_acc = mean([t.nn_preds.wh == t.theta.wh for t in trials])
    
    exact_match = mean([t.nn_correct for t in trials])
    
    hamming_acc = mean([
        sum([
            t.nn_preds.hd == t.theta.hd,
            t.nn_preds.sd == t.theta.sd,
            t.nn_preds.ao == t.theta.ao,
            t.nn_preds.wh == t.theta.wh
        ]) / 4.0
        for t in trials
    ])
    
    return (
        per_param = (hd=hd_acc, sd=sd_acc, ao=ao_acc, wh=wh_acc),
        mean_param = mean([hd_acc, sd_acc, ao_acc, wh_acc]),
        exact_match = exact_match,
        hamming = hamming_acc
    )
end

function get_learning_curves_plot(all_trials, n_sentences)
    n_trials = length(all_trials)
    naive_curves = [t.naive_metrics.target_prob_history for t in all_trials]
    guided_curves = [t.guided_metrics.target_prob_history for t in all_trials]
    
    avg_naive = [mean([c[i] for c in naive_curves]) for i in 1:n_sentences]
    avg_guided = [mean([c[i] for c in guided_curves]) for i in 1:n_sentences]
    
    p = plot(1:n_sentences, avg_naive, label="Naive", 
             linewidth=2, color=:indianred, ribbon=std.([[c[i] for c in naive_curves] for i in 1:n_sentences])/sqrt(n_trials), fillalpha=0.2)
    plot!(p, 1:n_sentences, avg_guided, label="Guided", 
          linewidth=2, color=:steelblue, ribbon=std.([[c[i] for c in guided_curves] for i in 1:n_sentences])/sqrt(n_trials), fillalpha=0.2)
    plot!(p, xlabel="Sentences", ylabel="P(correct)", title="Overall Learning Curves", legend=:bottomright)
    return p
end

function get_survival_plot(all_trials, n_sentences)
    naive_survival = [mean([t.naive_metrics.sentences_to_50 <= i for t in all_trials]) for i in 1:n_sentences]
    guided_survival = [mean([t.guided_metrics.sentences_to_50 <= i for t in all_trials]) for i in 1:n_sentences]
    
    p = plot(1:n_sentences, naive_survival, label="Naive", linewidth=2, color=:indianred)
    plot!(p, 1:n_sentences, guided_survival, label="Guided", linewidth=2, color=:steelblue)
    plot!(p, xlabel="Sentences", ylabel="Fraction > 50%", title="Convergence Speed", legend=:bottomright)
    return p
end

function get_nn_accuracy_bar_plot(nn_metrics)
    param_labels = ["Head\nDir", "Subj\nDrop", "Adj\nOrder", "Wh-\nFront"]
    param_accs = [nn_metrics.per_param.hd, nn_metrics.per_param.sd, 
                  nn_metrics.per_param.ao, nn_metrics.per_param.wh]
    
    p = bar(param_labels, param_accs, title="NN Param Accuracy",
            ylabel="Accuracy", legend=false, color=:steelblue, fillalpha=0.7, ylim=(0,1.1))
    hline!(p, [0.5], linestyle=:dash, color=:gray)
    return p
end

function get_language_difficulty_plot(all_trials)
    trials_by_lang = Dict()
    for t in all_trials
        key = (t.theta.hd, t.theta.sd, t.theta.ao, t.theta.wh)
        if !haskey(trials_by_lang, key) trials_by_lang[key] = [] end
        push!(trials_by_lang[key], t)
    end
    
    labels = String[]
    probs = Float64[]
    
    for (theta, trials) in trials_by_lang
        l = (theta[1] ? "VO" : "OV") * (theta[2] ? "-Drop" : "-Keep")
        push!(labels, l)
        push!(probs, mean([t.guided_metrics.final_target_prob for t in trials]))
    end
    
    order = sortperm(probs)
    
    p = bar(labels[order], probs[order], orientation=:v, xrotation=45,
            title="Difficulty by Lang Type", ylabel="Final P(correct)", legend=false, color=:steelblue)
    return p
end

function get_convergence_speed_plot(all_trials, n_sentences)
    trials_by_lang = Dict()
    for t in all_trials
        key = (t.theta.hd, t.theta.sd, t.theta.ao, t.theta.wh)
        if !haskey(trials_by_lang, key) trials_by_lang[key] = [] end
        push!(trials_by_lang[key], t)
    end
    
    labels = String[]
    naive_speed = Float64[]
    guided_speed = Float64[]
    
    for (theta, trials) in trials_by_lang
        l = (theta[1] ? "VO" : "OV") * (theta[2] ? "D" : "K") * (theta[3] ? "A" : "N") * (theta[4] ? "F" : "S")
        push!(labels, l)
        push!(naive_speed, mean([min(t.naive_metrics.sentences_to_90, n_sentences) for t in trials]))
        push!(guided_speed, mean([min(t.guided_metrics.sentences_to_90, n_sentences) for t in trials]))
    end
    
    order = sortperm(guided_speed)
    
    p = groupedbar(labels[order], hcat(naive_speed[order], guided_speed[order]),
                   label=["Naive" "Guided"], title="Sentences to 90% Conf.",
                   xrotation=45, color=[:indianred :steelblue], legend=:topleft)
    return p
end

function get_typology_curve_plot(all_trials, n_sentences)
    natural_patterns = [
        (hd=true, sd=false, ao=true, wh=true), (hd=false, sd=true, ao=false, wh=false),
        (hd=true, sd=true, ao=false, wh=true), (hd=false, sd=false, ao=false, wh=false)
    ]
    natural_trials = [t for t in all_trials if t.theta in natural_patterns]
    rare_trials = [t for t in all_trials if !(t.theta in natural_patterns)]
    
    nat_curves = [t.guided_metrics.target_prob_history for t in natural_trials]
    rare_curves = [t.guided_metrics.target_prob_history for t in rare_trials]
    
    avg_nat = [mean([c[i] for c in nat_curves]) for i in 1:n_sentences]
    avg_rare = [mean([c[i] for c in rare_curves]) for i in 1:n_sentences]
    
    p = plot(1:n_sentences, avg_nat, label="Natural", linewidth=3, color=:forestgreen)
    plot!(p, 1:n_sentences, avg_rare, label="Rare", linewidth=3, color=:darkorange)
    plot!(p, xlabel="Sentences", ylabel="P(correct)", title="Typology: Learning Curves", legend=:bottomright)
    return p
end

function get_typology_bar_plot(all_trials)
    natural_patterns = [
        (hd=true, sd=false, ao=true, wh=true),   # English
        (hd=false, sd=true, ao=false, wh=false), # Japanese
        (hd=true, sd=true, ao=false, wh=true),   # Spanish
        (hd=false, sd=false, ao=false, wh=false), # German-ish
    ]
    natural_trials = [t for t in all_trials if t.theta in natural_patterns]
    rare_trials = [t for t in all_trials if !(t.theta in natural_patterns)]
    
    nat_acc = mean([t.guided_metrics.final_target_prob for t in natural_trials])
    rare_acc = mean([t.guided_metrics.final_target_prob for t in rare_trials])
    
    p = bar(["Natural", "Rare"], [nat_acc, rare_acc], 
            title="Natural vs Rare", color=[:forestgreen, :darkorange], legend=false, ylim=(0,1))
    return p
end

function get_param_difficulty_summary_plot(all_trials)
    params = [:hd, :sd, :ao, :wh]
    param_names = ["HD", "SD", "AO", "Wh"]
    
    vals_true = Float64[]
    vals_false = Float64[]
    
    for param in params
        true_trials = [t for t in all_trials if getfield(t.theta, param) == true]
        false_trials = [t for t in all_trials if getfield(t.theta, param) == false]
        push!(vals_true, mean([t.guided_metrics.final_target_prob for t in true_trials]))
        push!(vals_false, mean([t.guided_metrics.final_target_prob for t in false_trials]))
    end
    
    p = groupedbar(param_names, hcat(vals_false, vals_true), 
        label=["Param=False" "Param=True"], title="Param Learnability", 
        color=[:indianred :steelblue], legend=:bottomright, ylim=(0,1))
    return p
end

function get_combined_stats_text_plot(all_trials, nn_metrics)
    naive_auc = mean([t.naive_metrics.auc for t in all_trials])
    guided_auc = mean([t.guided_metrics.auc for t in all_trials])
    naive_final = mean([t.naive_metrics.final_target_prob for t in all_trials])
    guided_final = mean([t.guided_metrics.final_target_prob for t in all_trials])
    
    p = plot(axis=false, grid=false, ticks=false, xlim=(0,1), ylim=(0,1))
    
    # SMC Stats
    annotate!(p, 0.5, 0.95, text("SMC METRICS", :center, 11))
    annotate!(p, 0.1, 0.85, text("Naive Final P:", :left, 9))
    annotate!(p, 0.9, 0.85, text("$(round(naive_final*100, digits=1))%", :right, :indianred, 9))
    annotate!(p, 0.1, 0.75, text("Guided Final P:", :left, 9))
    annotate!(p, 0.9, 0.75, text("$(round(guided_final*100, digits=1))%", :right, :steelblue, 9))
    
    imp = (guided_auc - naive_auc)/naive_auc * 100
    annotate!(p, 0.5, 0.65, text("AUC Improvement: +$(round(imp, digits=1))%", :center, 9))
    
    annotate!(p, 0.5, 0.55, text("─"^40, :center, 10))
    
    # NN Stats
    annotate!(p, 0.5, 0.45, text("NN DIAGNOSTICS", :center, 11))
    annotate!(p, 0.1, 0.35, text("Exact Match:", :left, 9))
    annotate!(p, 0.9, 0.35, text("$(round(nn_metrics.exact_match*100, digits=1))%", :right, 9))
    annotate!(p, 0.1, 0.25, text("Hamming Acc:", :left, 9))
    annotate!(p, 0.9, 0.25, text("$(round(nn_metrics.hamming*100, digits=1))%", :right, 9))
    
    return p
end

In [ ]:
function create_master_dashboard(all_trials)
    n_sentences = length(all_trials[1].naive_metrics.target_prob_history)
    nn_metrics = compute_nn_metrics(all_trials)

    p1 = get_learning_curves_plot(all_trials, n_sentences)
    p2 = get_survival_plot(all_trials, n_sentences)
    p3 = get_combined_stats_text_plot(all_trials, nn_metrics)
    
    p4 = get_nn_accuracy_bar_plot(nn_metrics)
    p5 = get_param_difficulty_summary_plot(all_trials)
    p6 = get_convergence_speed_plot(all_trials, n_sentences)
    
    p7 = get_language_difficulty_plot(all_trials)
    p8 = get_typology_curve_plot(all_trials, n_sentences)
    p9 = get_typology_bar_plot(all_trials)
    
    l = @layout [a b c; d e f; g h i]
    
    main_plot = plot(p1, p2, p3, p4, p5, p6, p7, p8, p9, 
                     layout=l, size=(1200, 1000), 
                     plot_title="Results Dashboard",
                     margin=4Plots.mm)
    
    return main_plot
end

In [ ]:
p1 = create_master_dashboard(all_trials)
display(p1)